# P7 — Observabilidad: Detección de Drift con Evidently

**RA/CE**: RA4c (evaluación de modelos de automatización industrial)
**Fase**: F7 — Observabilidad
**Teoría asociada**: `01-teoria/08_observabilidad.md`

---

## Objetivos de Aprendizaje

1. Detectar data drift y concept drift con Evidently
2. Generar dashboards HTML de monitorización interpretables
3. Configurar alertas programáticas basadas en tests estadísticos
4. Conectar la monitorización con el resto del stack convergente (RA4c)

## Prerrequisitos

- [ ] Concepto de drift en modelos de ML (teoría F7)
- [ ] Evidently instalado: `pip install evidently`
- [ ] pandas, numpy, scikit-learn instalados
- [ ] Finalizadas las prácticas P5 y P6 (opcional, usaremos datos sintéticos)

## Contexto

En las prácticas anteriores construiste un clasificador de incidencias, lo
serviste como API (P5) y creaste agentes inteligentes (P6). Pero... ¿cómo
sabes si el modelo sigue funcionando bien en producción?

Los datos cambian. Los usuarios cambian su forma de escribir incidencias.
El modelo se degrada silenciosamente. En esta práctica aprenderás a detectar
ese deterioro antes de que los usuarios se quejen.

**Caso**: Monitorizamos nuestro clasificador de incidencias técnicas.
Simularemos datos de entrenamiento (referencia) y datos de producción
(con drift artificial) para ver cómo Evidently los detecta.

---
## Parte 1: Verifica tu entorno

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

HAS_DEPS = True
missing = []

try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset
    print('✅ Evidently Report OK')
except ImportError as e:
    HAS_DEPS = False
    missing.append('evidently')
    print('❌ Missing: evidently')

try:
    from evidently.test_suite import TestSuite
    from evidently.tests import TestShareOfDriftedColumns
    print('✅ Evidently TestSuite OK')
except ImportError:
    HAS_DEPS = False
    if 'evidently' not in missing:
        missing.append('evidently (tests)')
    print('❌ Missing: evidently test suite')

try:
    import pandas as pd
    import numpy as np
    print('✅ Data dependencies OK')
except ImportError as e:
    HAS_DEPS = False
    mod = str(e).split("'")[1] if "'" in str(e) else str(e)
    missing.append(mod)
    print(f'❌ Missing: {mod}')

if not HAS_DEPS:
    print(f'\nInstall: pip install {" ".join(missing)}')
else:
    print('\n✅ Environment ready')

---
## Parte 2: Crear Datos Sintéticos de Referencia y Producción

Simularemos un clasificador de incidencias con 6 features numéricas.
Crearemos dos conjuntos:
- **Referencia**: datos de entrenamiento (distribución original)
- **Actual**: datos de producción con drift artificial en algunas features

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n_ref = 1000
n_cur = 500

# Datos de referencia (entrenamiento)
ref = pd.DataFrame({
    'text_length': np.random.normal(120, 30, n_ref).clip(10, 500),
    'num_words': np.random.poisson(25, n_ref).clip(1, 100),
    'has_urgency': np.random.binomial(1, 0.3, n_ref),
    'hour_of_day': np.random.randint(0, 24, n_ref),
    'num_attachments': np.random.poisson(1, n_ref).clip(0, 10),
    'priority_score': np.random.exponential(3, n_ref).clip(0, 10),
    'target': np.random.choice(['red', 'servidor', 'software', 'hardware', 'seguridad'], n_ref)
})

# Datos actuales (producción) — con drift en algunas features
cur = pd.DataFrame({
    'text_length': np.random.normal(200, 50, n_cur).clip(10, 500),  # ← DRIFT: media más alta
    'num_words': np.random.poisson(40, n_cur).clip(1, 100),         # ← DRIFT: más palabras
    'has_urgency': np.random.binomial(1, 0.5, n_cur),               # ← DRIFT: más urgencia
    'hour_of_day': np.random.randint(0, 24, n_cur),                 # ← Sin drift
    'num_attachments': np.random.poisson(1, n_cur).clip(0, 10),    # ← Sin drift
    'priority_score': np.random.exponential(3, n_cur).clip(0, 10), # ← Sin drift
    'target': np.random.choice(['red', 'servidor', 'software', 'hardware', 'seguridad'], n_cur)
})

print(f'Referencia: {ref.shape[0]} muestras, {ref.shape[1]} features')
print(f'Actual:     {cur.shape[0]} muestras, {cur.shape[1]} features')
print()
print('Distribución de text_length (referencia):', 
      f'media={ref["text_length"].mean():.1f}, std={ref["text_length"].std():.1f}')
print('Distribución de text_length (actual):    ',
      f'media={cur["text_length"].mean():.1f}, std={cur["text_length"].std():.1f}')

---
## Parte 3: Detectar Data Drift con Evidently

Vamos a configurar Evidently para comparar los datos de referencia con
los datos actuales usando tests estadísticos.

In [ ]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

# Configurar reporte de drift
# KS test para numéricas, PSI para categóricas
drift_report = Report(metrics=[
    DataDriftPreset(
        stattest='ks',           # Kolmogorov-Smirnov para numéricas
        cat_stattest='psi',      # Population Stability Index para categóricas
        drift_share=0.3          # Umbral: >30% features con drift = alerta
    )
])

drift_report.run(reference_data=ref, current_data=cur)

# Extraer resultados como dict
results = drift_report.as_dict()

drifted_features = results['metrics'][0]['result']['drift_by_features']
total_features = len(drifted_features)
drifted_count = sum(1 for f in drifted_features.values() if f['drift_detected'])

print(f'Features totales: {total_features}')
print(f'Features con drift detectado: {drifted_count}')
print(f'Porcentaje de drift: {drifted_count/total_features*100:.1f}%')
print()

for feat, info in drifted_features.items():
    status = '⚠️ DRIFT' if info['drift_detected'] else '✅ OK'
    print(f'{status}: {feat} (p-value: {info.get("p_value", "N/A"):.4f})')

### Interpretación

- **text_length** debería mostrar drift porque cambiamos su distribución
- **hour_of_day** debería estar OK porque mantuvimos la distribución

Pregunta: ¿Qué features esperas que tengan drift? ¿Coincide con la realidad?

In [ ]:
# Guardar reporte como HTML
drift_report.save_html('reports/drift_report_p7.html')
print('✅ Reporte guardado: reports/drift_report_p7.html')
print('Ábrelo en el navegador para ver el dashboard interactivo.')

---
## Parte 4: Tests de Hipótesis y Alertas

Evidently TestSuite permite configurar alertas programáticas: si el test
falla, podemos disparar acciones automáticas (re-entrenamiento, notificación).

In [ ]:
from evidently.test_suite import TestSuite
from evidently.tests import TestShareOfDriftedColumns

# Configurar alerta: si >30% de features tienen drift, FALLA
drift_alarm = TestSuite(tests=[
    TestShareOfDriftedColumns(gte=0.3)  # Alarma si >= 30%
])

drift_alarm.run(reference_data=ref, current_data=cur)
test_results = drift_alarm.as_dict()

status = test_results['tests'][0]['status']
description = test_results['tests'][0]['description']

print(f'Test: {description}')
print(f'Status: {status}')

if status == 'FAILED':
    print('\n⚠️ ALERTA: Se ha detectado drift significativo en los datos.')
    print('Acción recomendada: disparar re-entrenamiento del modelo.')

In [ ]:
# Simular alerta con acción
if status == 'FAILED':
    import json
    alert = {
        'type': 'drift_alert',
        'severity': 'warning',
        'drift_ratio': drifted_count / total_features,
        'action': 'trigger_retraining',
        'timestamp': pd.Timestamp.now().isoformat()
    }
    print('🚨 Alerta generada:')
    print(json.dumps(alert, indent=2))
    print('\nEn producción, esta alerta enviaría un webhook a Prefect para re-entrenar.')
else:
    print('✅ Sin drift significativo. El modelo sigue siendo válido.')

---
## Parte 5: Drift en la Predicción (Concept Drift Simulado)

Ahora simularemos concept drift: la relación entre features y target cambia.
El modelo ve los mismos datos de entrada pero la distribución de las
predicciones cambia drásticamente.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Preparar datos
feature_cols = ['text_length', 'num_words', 'has_urgency', 'hour_of_day',
                'num_attachments', 'priority_score']

X_ref = ref[feature_cols]
le = LabelEncoder()
y_ref = le.fit_transform(ref['target'])

# Entrenar modelo con datos de referencia
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_ref, y_ref)

# Predecir sobre datos actuales
X_cur = cur[feature_cols]
pred_cur = model.predict(X_cur)
pred_proba_cur = model.predict_proba(X_cur).max(axis=1)

# Añadir predicciones al dataset actual
cur_with_pred = cur.copy()
cur_with_pred['predicted_class'] = le.inverse_transform(pred_cur)
cur_with_pred['confidence'] = pred_proba_cur

print('Distribución de predicciones:')
print(cur_with_pred['predicted_class'].value_counts())
print(f'\nConfianza media: {cur_with_pred["confidence"].mean():.3f}')

### Analizar Predicciones vs. Baseline

Comparamos la distribución de clases predichas entre referencia y actual:

In [ ]:
from evidently.metric_preset import TargetDriftPreset

# Reporte de drift en la variable objetivo (target)
target_report = Report(metrics=[
    TargetDriftPreset()
])

target_report.run(reference_data=ref.assign(target=le.transform(ref['target'])),
                  current_data=cur.assign(target=le.transform(cur['target'])))

target_results = target_report.as_dict()
print('Drift en variable target:')
for metric in target_results['metrics']:
    if 'drift_detected' in str(metric):
        drift_status = metric.get('result', {}).get('drift_detected', 'unknown')
        print(f'  Drift detectado: {drift_status}')

target_report.save_html('reports/target_drift_p7.html')
print('\n✅ Reporte de target drift guardado: reports/target_drift_p7.html')

---
## Parte 6: Conexión con el Stack Convergente

La monitorización no es una fase aislada. Conecta con:

- **F2 (Pipeline)**: ¿los datos de hoy se parecen a los de entrenamiento?
- **F3 (MLflow)**: ¿la accuracy del modelo en producción coincide con la del registro?
- **F4 (Prefect)**: ¿podemos disparar un re-entrenamiento automático cuando hay drift?
- **F5 (FastAPI)**: ¿la latencia de la API aumenta porque los datos han cambiado?
- **F6 (Agentes)**: ¿los agentes siguen respondiendo correctamente con los nuevos datos?

In [ ]:
# Resumen de conexiones
print('Conexiones del stack convergente detectadas:')
print('  F2 → F7: Evidently compara datos actuales vs. referencia de entrenamiento')
print('  F3 → F7: El modelo registrado en MLflow es el que se monitoriza')
print('  F4 → F7: Prefect puede orquestar la generación periódica de reportes')
print('  F5 → F7: La API registra predicciones que Evidently analiza')
print('  F6 → F7: Los agentes alimentan métricas de faithfulness a Evidently')
print()
print('Conexión RA4c: la monitorización automatizada permite evaluar si los')
print('modelos de automatización cumplen los resultados esperados en producción.')

---
## Parte 7: Análisis y Reflexión

Responde a las siguientes preguntas en la celda de código inferior:

In [ ]:
analisis = '''
1. ¿Qué features detectaron drift y por qué?
   
   [Escribe aquí tu análisis]

2. ¿Cómo afectaría este drift a la calidad del clasificador de incidencias?
   
   [Escribe aquí tu análisis]

3. ¿Qué acción tomarías si este drift se detectara en producción?
   
   [Escribe aquí tu análisis]

4. ¿Cómo se conecta esta práctica con RA4c?
   
   [Escribe aquí tu análisis]

5. ¿Qué métrica adicional monitorizarías para tener un cuadro completo?
   
   [Escribe aquí tu análisis]
'''
print(analisis)

---
## Resumen de la Práctica

✅ Has detectado data drift con Evidently usando tests estadísticos
✅ Has generado un dashboard HTML interpretable con el reporte de drift
✅ Has configurado alertas programáticas basadas en tests de hipótesis
✅ Has simulado concept drift y analizado su impacto
✅ Has conectado la monitorización con el resto del stack convergente
✅ Has analizado cómo la observabilidad automatizada (RA4c) permite evaluar
   si los modelos de automatización cumplen los resultados esperados